# Development workflows on Databricks

**Purpose:** There are multiple ways of writing and running code on Databricks, each with their own advantages. This notebook describes these approaches and gives a rule for when to use each. 

**What this compares.**
1. Interactive workspace: notebooks attached to a cluster, in the browser.
2. Local IDE + Databricks Connect: VS Code and your editor's tooling, remote Spark.
3. Asset Bundle deploy: versioned jobs and pipelines shipped via `databricks bundle`. This will also be covered in more detail in `02_asset_bundles`

The point of all three is that one source file can travel the whole way:

1. Sketch it in a workspace notebook.
2. Pull it local with the VS Code extension, refactor logic into `src/`, add `pytest`, and run against remote Spark via Connect.
3. Reference it from a job in `databricks.yml` and deploy the bundle. The same file now runs as a governed, scheduled production job.

| Dimension | Interactive workspace | VS Code + Connect | Asset Bundle deploy |
|---|---|---|---|
| Speed to first result | **Fastest** | Medium | Slowest to set up |
| Reproducibility | Low | Medium | **High** |
| Governance / audit | Low | Medium | **High** |
| Git + code review | Awkward | **Native** | **Native** |
| Unit testing | Hard | **Easy** | **Easy** |
| Best for | Exploration, demos | Engineering, tests | Production, CI/CD |

Rule of thumb: explore in the workspace, build in the IDE, ship with bundles. If a piece of code has outlived the day it was written, it belongs at least one rung down.

```mermaid
graph LR
    A["Workspace notebook: explore fast"] -->|outlives the day| B["VS Code + Connect: add git + tests"]
    B -->|needs to ship| C["Asset Bundle: versioned, scheduled job"]
    A -.->|one source file travels the whole way| C
```

## Compute strategy: match the cluster to the workload

| Compute | Reach for it when | Cost shape | The catch |
|---|---|---|---|
| Serverless | Ad-hoc queries, prototyping, instant-on | Pay per active execution | No init scripts or custom cluster config |
| All-purpose | Connected IDE (Connect) or live notebook authoring | Accrues continuously while running | Priciest per DBU; easy to leave running |
| Job compute | Scheduled, long, or reproducible runs | Cheapest DBUs, ephemeral | Cold start spins up per run |

Default to the **Databricks Runtime for ML (DBR ML)**: it ships tuned MLflow, PyTorch, and distributed-training libraries, so you skip most `pip install` friction. Use a **single node** for `pandas`/`scikit-learn`; reach for **multi-node** only when the work is genuinely PySpark or distributed training. Need cluster scale without rewriting pure Python? Bridge with the [Pandas API on Spark](https://docs.databricks.com/en/pandas/pandas-function-apis.html).


## (a) Interactive workspace: explore fast

**Use it when** you're exploring a dataset, prototyping a model, debugging interactively, or giving a demo. The feedback loop is the shortest you'll get.

**Notes:**
- **Ephemeral state:** The REPL environment is incredibly fast for iteration, but hidden cell state is the enemy of reproducibility. Always restart the kernel and "run all" before trusting your results.
- **The technical debt trap:** Notebooks are notoriously difficult to properly modularize, lint, or unit-test. The moment core logic proves valuable, refactor it out of the notebook and into an importable Python package.
- **Version control limits:** Databricks Git Folders are sufficient for basic syncing, but the browser UI lacks the robust tooling needed to handle complex merge conflicts, rebasing, or multi-branch tracking. On collaborative enterprise projects, relying exclusively on the workspace UI creates severe friction; transitioning to a local IDE with native `.git` management becomes mandatory.

In [0]:
# --- 1. STATE MANAGEMENT ---
# The workspace REPL hides state. If things act weird, uncomment and run this to nuke the Python process.
# dbutils.library.restartPython()

# If you are actively developing local packages (e.g., in our src/ directory) via workspace files,
# autoreload forces the notebook to pick up your module changes without restarting the cluster.
%load_ext autoreload
%autoreload 2

# --- 2. PARAMETERIZATION ---
# Never hardcode environment strings or limits. 
# dbutils.widgets creates UI inputs at the top of the notebook so anyone can toggle them safely.
dbutils.widgets.dropdown("env", "dev", ["dev", "staging", "prod"], "Environment")
dbutils.widgets.text("sample_size", "1000", "Churn Sample Size")

current_env = dbutils.widgets.get("env")
n_samples = int(dbutils.widgets.get("sample_size"))

print(f"Target Environment: {current_env.upper()} | Generating {n_samples} synthetic churn records...")

# --- 3. INTERACTIVE EXPLORATION (Customer Churn Spine) ---
# In Phase 5, we abstract this cleanly into `src.utils.data.make_churn_data()`. 
# For quick workspace exploration, we can mock a PySpark dataframe instantly.
from pyspark.sql.functions import rand, expr

df_churn_mock = (
    spark.range(n_samples)
    .withColumn("customer_id", expr("uuid()"))
    .withColumn("tenure_months", (rand() * 72).cast("int"))
    .withColumn("monthly_charges", (rand() * 100 + 20).cast("int"))
    # Simple synthetic signal: short tenure + high cost = high churn risk
    .withColumn("churn_label", expr("CASE WHEN tenure_months < 12 AND monthly_charges > 80 THEN 1 ELSE 0 END"))
)

# display() is the superpower of the interactive workspace. 
# It provides instant data profiling, distributions, and visualization without matplotlib boilerplate.
display(df_churn_mock)


## (b) Local IDE + Databricks Connect: engineer properly

**Use it when** the code actually matters. If a script requires unit tests, rigorous peer review, or a production schedule, it must be engineered here.

**The Engineering Standard:**
- **Native Git:** Solves the workspace Git Folders ceiling. Use a real terminal for multi-branch workflows, rebasing, and complex conflict resolution.
- **Modularization:** Break monolithic notebooks apart. Core logic belongs in an importable Python package (e.g., `src/`), not hidden in notebook cell state.
- **CI/CD Tooling:** Unlocks standard local workflows: `pytest`, linters (`ruff`, `flake8`), type-checking (`mypy`), and pre-commit hooks.

**Authentication:** Never hardcode tokens. Prefer OAuth user-to-machine over legacy Personal Access Tokens (PATs). Run this locally to configure your `~/.databrickscfg` profile for both the IDE and the CLI:

```bash
databricks auth login --host https://<your-workspace-host>
# creates or updates a named profile; the VS Code extension and CLI both read it
```

In [ ]:
# "Where am I running?" - the theme of this notebook, made executable.
# Runs anywhere: workspace, local IDE via Connect, or plain Python (CI / offline).
def where_am_i() -> str:
    import os

    if "DATABRICKS_RUNTIME_VERSION" in os.environ:
        return "Interactive workspace (notebook on a cluster)"
    try:
        import databricks.connect  # noqa: F401

        return "Local IDE + Databricks Connect (remote Spark)"
    except ImportError:
        return "Plain Python, no Spark - e.g. CI or offline"


print("Running in:", where_am_i())

# Databricks Connect sanity check: guarded so it never hard-fails.
# In a workspace notebook you already have a `spark` session and don't need this.
# From a local IDE, Connect builds a session that runs on a remote cluster.
try:
    from databricks.connect import DatabricksSession

    spark = DatabricksSession.builder.getOrCreate()
    print("Remote Spark reachable:", spark.range(3).count(), "rows computed remotely.")
except Exception as exc:  # ImportError locally, or auth/config errors
    print(f"Databricks Connect not active here ({type(exc).__name__}).")
    print("Expected on GitHub or without a configured profile.")
    print("Set up: databricks auth login, then point the profile at a cluster or serverless.")

## (c) Asset Bundle deploy: ship to production

A [Databricks Asset Bundle](https://docs.databricks.com/dev-tools/bundles/index.html) packages your notebooks, jobs, and pipelines as versioned, declarative config (`databricks.yml`) that deploys identically to dev, staging, and prod. No clicking in the UI; the workspace state becomes a function of code in git.

**Use it when** the work needs to run on a schedule, move across environments, or pass through CI/CD. This is the production destination, covered in depth in notebook 02.

```bash
databricks bundle validate -t dev      # check config against the schema
databricks bundle deploy   -t staging  # create/update jobs + pipelines in staging
databricks bundle run churn_training -t staging
```

*(Shown for shape, not executed here.)*

**Jobs & Pipelines:**
- **The entry point:** You can schedule a workspace notebook directly via the Databricks Workflows UI. 
- **The ceiling:** While perfectly fine for an isolated, ad-hoc data pull, UI-driven "click-ops" orchestration creates a fragmented, unversioned mess at scale. Avoid relying on UI-scheduled workspace notebooks for any critical ML pipelines.

## Further reading

[`docs/links.md`](../docs/links.md) carries the full annotated list, one line of *why* per entry. The essentials for this notebook:

- [Databricks extension for VS Code](https://docs.databricks.com/dev-tools/vscode-ext/index.html)
- [Databricks Connect](https://docs.databricks.com/dev-tools/databricks-connect/index.html)
- [Authentication (OAuth / profiles)](https://docs.databricks.com/dev-tools/auth/index.html)
- [Asset Bundles](https://docs.databricks.com/dev-tools/bundles/index.html)